# 01 — Il dataset solo politici, con più testate (tentativo 12)

Gli articoli puliti (`1-data-cleaning/01`, sui due giri di ottobre **più il terzo giro del 25 ottobre**) delle testate
che per NewsGuard **2026** hanno un punteggio e **un solo argomento, "Political news or commentary"**; etichetta = il
livello NewsGuard 2026. Fissato il 25 ottobre, prima dei risultati. Regole del tentativo 11:

1. **gruppi** = le nostre reti (`1-data-cleaning/02_reti`, calcolate su tutte le testate scaricate) ∪ stesso owner ∪
   stesso Parent Domain, presi dal 2026 e (dove c'è) dal 2022; i gruppi si fanno su tutte le testate, poi si tengono
   solo le politiche;
2. **copie tra gruppi diversi**: di un testo pubblicato da testate politiche di gruppi diversi si tiene la copia più vecchia;
3. **valutate** (il test): almeno 20 articoli e al più 5 testate per gruppo, a caso con seed fisso, senza guardare le
   etichette; **di solo training**: le altre con almeno 5 articoli, peso min(1, 5/n);
4. **fold**: 5, per gruppo, stratificati sul livello 2026.

Uscite: `Dataset/versioni/dataset.csv` e `fold.csv`; in `Results/2-dataset` passi e composizione, con quante testate
vengono da ottobre e quante dal terzo giro.

In [1]:
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from comune import COPIE, LIVELLI, NG2022, NG2026, PULITI, RESULTS_DATASET, RETI, SEED, VERSIONI, leggi_newsguard, livello, normalizza_owner

MIN_ARTICOLI = 20
MAX_TESTATE_PER_GRUPPO = 5
MIN_ARTICOLI_TRAINING = 5
RESULTS_DATASET.mkdir(parents=True, exist_ok=True); VERSIONI.mkdir(parents=True, exist_ok=True)
n26, n22 = leggi_newsguard(NG2026), leggi_newsguard(NG2022)
ottobre = set(pd.read_csv("../Dataset/NuovoScraping_ottobre/elenco_testate.csv").dominio.str.lower())
art = pd.read_csv(PULITI, parse_dates=["data"])
passi = []


def passo(nome):
    passi.append({"passo": nome, "articoli": len(art), "testate": art.domain.nunique()})
    print(f"{nome}: {len(art)} articoli, {art.domain.nunique()} testate")


passo("articoli puliti (ottobre + terzo giro)")
argomenti = art.domain.map(n26.Topics).fillna("").str.split(",").map(lambda x: [a.strip() for a in x if a.strip()])
art = art[art.domain.map(n26.Score).notna() & argomenti.map(lambda x: x == ["Political news or commentary"]).to_numpy()]
passo("1. testate con un punteggio 2026 e il solo argomento politico")

articoli puliti (ottobre + terzo giro): 143106 articoli, 1816 testate


1. testate con un punteggio 2026 e il solo argomento politico: 43728 articoli, 532 testate


In [2]:
reti = pd.read_csv(RETI).set_index("domain")
nodi = sorted(set(reti.index) | set(art.domain))
G = nx.Graph(); G.add_nodes_from(nodi)
for _, g in reti.groupby("rete"):
    G.add_edges_from(zip(g.index, g.index[1:]))
for ng in (n22, n26):
    for campo, norma in [("Owner", normalizza_owner), ("Parent Domain", lambda x: "" if str(x) == "nan" else str(x).strip().lower())]:
        s = pd.Series({d: norma(ng[campo].get(d, "")) if d in ng.index else "" for d in nodi})
        s = s[s != ""]
        for _, g in s.groupby(s):
            G.add_edges_from(zip(g.index, g.index[1:]))
gruppo = {d: i for i, c in enumerate(sorted(nx.connected_components(G), key=len, reverse=True)) for d in c}
art["gruppo"] = art.domain.map(gruppo); art["rete"] = art.domain.map(reti.rete)
print(f"{len(set(gruppo.values()))} gruppi in tutto; le testate politiche stanno in {art.gruppo.nunique()} gruppi")

541 gruppi in tutto; le testate politiche stanno in 251 gruppi


In [3]:
copie = pd.read_csv(COPIE)
presenti = set(art.article_id)
copie = copie[copie.articolo_a.isin(presenti) & copie.articolo_b.isin(presenti)]
C = nx.Graph(); C.add_edges_from(zip(copie.articolo_a, copie.articolo_b))
info = art.set_index("article_id")
via = set()
for fam in nx.connected_components(C):
    fam = sorted(fam, key=lambda x: (info.data[x], x))
    if len({info.gruppo[x] for x in fam}) > 1:
        primo = info.gruppo[fam[0]]
        via |= {x for x in fam if info.gruppo[x] != primo}
art = art[~art.article_id.isin(via)]
passo("2. senza copie tra gruppi diversi")
n = art.groupby("domain").size()
art = art[art.domain.map(n) >= MIN_ARTICOLI_TRAINING]
passo(f"3. testate con almeno {MIN_ARTICOLI_TRAINING} articoli (training)")

2. senza copie tra gruppi diversi: 43530 articoli, 532 testate
3. testate con almeno 5 articoli (training): 43442 articoli, 495 testate


In [4]:
t = art.groupby("domain").agg(gruppo=("gruppo", "first"), rete=("rete", "first"), articoli=("article_id", "size"))
idonee = t[t.articoli >= MIN_ARTICOLI].reset_index()
valutate = set(idonee.sample(frac=1, random_state=SEED).groupby("gruppo").head(MAX_TESTATE_PER_GRUPPO).domain)
t["valutazione"] = t.index.isin(valutate)
t["testate_nel_gruppo"] = t.groupby("gruppo").articoli.transform("size")
t["peso"] = np.minimum(1.0, MAX_TESTATE_PER_GRUPPO / t.testate_nel_gruppo)
t["score"], t["score_2022"] = t.index.map(n26.Score), t.index.map(n22.Score)
t["label"], t["label_2022"] = livello(t.score.to_numpy()).to_numpy(), livello(t.score_2022.to_numpy()).to_numpy()
t["giro"] = np.where(t.index.isin(ottobre), "ottobre", "terzo giro")
passi.append({"passo": f"4. valutate: almeno {MIN_ARTICOLI} articoli, al più {MAX_TESTATE_PER_GRUPPO} per gruppo",
              "articoli": int(art.domain.isin(valutate).sum()), "testate": len(valutate)})
t = t.sort_index()
val = t[t.valutazione].reset_index()
val["fold_gruppo"] = 0
for i, (_, idx) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(val, val.label, val.gruppo), 1):
    val.loc[idx, "fold_gruppo"] = i
fold_del_gruppo = val.groupby("gruppo").fold_gruppo.first()
rng = np.random.default_rng(SEED)
senza = sorted(set(t.gruppo) - set(fold_del_gruppo.index))
fold_del_gruppo = pd.concat([fold_del_gruppo, pd.Series(rng.integers(1, 6, len(senza)), index=senza)])
t["fold_gruppo"] = t.gruppo.map(fold_del_gruppo).astype(int)
assert t.groupby("gruppo").fold_gruppo.nunique().max() == 1
pd.crosstab(val.fold_gruppo, val.label.map(dict(enumerate(LIVELLI))), margins=True, margins_name="tot")

label,l1,l2,l3,l4,l5,tot
fold_gruppo,,,,,,
1,9,11,7,16,8,51
2,10,11,6,16,8,51
3,9,11,6,17,8,51
4,9,11,6,17,8,51
5,9,11,6,16,8,50
tot,46,55,31,82,40,254


In [5]:
for c in ["label", "label_2022", "score", "valutazione", "peso", "fold_gruppo"]:
    art[c] = art.domain.map(t[c])
art[["article_id", "domain", "gruppo", "rete", "label", "label_2022", "score", "valutazione", "peso", "fold_gruppo",
     "data", "titolo", "autore", "url", "text"]].to_csv(VERSIONI / "dataset.csv", index=False)
t.reset_index(names="domain").to_csv(VERSIONI / "fold.csv", index=False)
pd.DataFrame(passi).set_index("passo").to_csv(RESULTS_DATASET / "01_passi.csv")
righe = []
for nome, sel in [("valutate", t.valutazione), ("tutte (training)", t.valutazione | True)]:
    for giro in ["tutti", "ottobre", "terzo giro"]:
        s = sel & ((t.giro == giro) if giro != "tutti" else True)
        r = {"testate": nome, "giro": giro, "n": int(s.sum()), "gruppi": int(t.gruppo[s].nunique()), "affidabili": float((t.label[s] >= 2).mean()) if s.sum() else np.nan}
        r.update(t.label[s].map(dict(enumerate(LIVELLI))).value_counts().reindex(LIVELLI).fillna(0).astype(int).to_dict())
        righe.append(r)
comp = pd.DataFrame(righe).set_index(["testate", "giro"]); comp.to_csv(RESULTS_DATASET / "01_composizione.csv")
display(pd.DataFrame(passi).set_index("passo")); comp.round(3)

,articoli,testate
passo,,
articoli puliti (ottobre + terzo giro),143106,1816
1. testate con un punteggio 2026 e il solo argomento politico,43728,532
2. senza copie tra gruppi diversi,43530,532
3. testate con almeno 5 articoli (training),43442,495
"4. valutate: almeno 20 articoli, al più 5 per gruppo",36068,254


n  gruppi  affidabili   l1  l2  l3   l4  l5
testate          giro                                                     
valutate         tutti       254     191       0.602   46  55  31   82  40
                 ottobre     140     109       0.607   18  37  20   44  21
                 terzo giro  114      98       0.596   28  18  11   38  19
tutte (training) tutti       495     230       0.368  215  98  37  101  44
                 ottobre     187     132       0.551   21  63  24   55  24
                 terzo giro  308     116       0.256  194  35  13   46  20